# 09 — Diffusion downscaling with uncertainty

CorrDiff-style conditional diffusion: a denoiser learns the distribution of **sharp** fields
conditioned on a **blurred** observation, and sampling it repeatedly yields an ensemble whose mean
is the prediction and whose spread is an uncertainty band. Here:

1. pairs of (blurred, sharp) Gaussian-mixture fields live on two Kratos model parts and are
   exported as matched grid series (`GridDatasetExportProcess` twice),
2. `CreateGridPairDataset` + `diffusion_utils.TrainDiffusionModel` run the conditional EDM loss
   over a tiny `EDMPrecondSuperResolution`/`SongUNet` — a physicsnemo `Module`, so it checkpoints
   as `.mdlus`,
3. `DiffusionInferenceProcess` deploys it in the loop: ensemble mean → `TEMPERATURE`, ensemble
   standard deviation → `NODAL_ERROR`.

Gaussians blur analytically (a blurred Gaussian is a wider Gaussian), so the ground truth is exact.


In [1]:
import math
from pathlib import Path

import numpy
import torch

import KratosMultiphysics as Kratos
from KratosMultiphysics.PhysicsNeMoApplication.processes.inference import diffusion_inference_process
from KratosMultiphysics.PhysicsNeMoApplication.training import diffusion_utils
from KratosMultiphysics.PhysicsNeMoApplication.processes.export import grid_dataset_export_process
from KratosMultiphysics.PhysicsNeMoApplication.training import training_utils
from KratosMultiphysics.PhysicsNeMoApplication.training.torch_dataset import CreateGridPairDataset

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

_HEX_TO_TETS = ((0, 1, 2, 6), (0, 2, 3, 6), (0, 3, 7, 6), (0, 7, 4, 6), (0, 4, 5, 6), (0, 5, 1, 6))

def create_tet_cube(model, name, divisions, variables):
    model_part = model.CreateModelPart(name)
    for variable in variables:
        model_part.AddNodalSolutionStepVariable(variable)
    props = model_part.CreateNewProperties(1)
    n = divisions + 1
    for i in range(n):
        for j in range(n):
            for k in range(n):
                model_part.CreateNewNode(i * n * n + j * n + k + 1,
                                         i / divisions, j / divisions, k / divisions)
    nid = lambda i, j, k: i * n * n + j * n + k + 1
    element_id = 0
    for i in range(divisions):
        for j in range(divisions):
            for k in range(divisions):
                corners = [nid(i, j, k), nid(i + 1, j, k), nid(i + 1, j + 1, k), nid(i, j + 1, k),
                           nid(i, j, k + 1), nid(i + 1, j, k + 1), nid(i + 1, j + 1, k + 1), nid(i, j + 1, k + 1)]
                for tet in _HEX_TO_TETS:
                    element_id += 1
                    model_part.CreateNewElement("Element3D4N", element_id, [corners[c] for c in tet], props)
    return model_part

BLUR = 0.012  # variance added by the "coarse observation" operator

def mixture(x, y, params, blur=0.0):
    """Two-Gaussian mixture; blur adds variance (exact Gaussian smoothing)."""
    value = 0.0
    for (cx, cy, var, amplitude) in params:
        total_var = var + blur
        value += amplitude * (var / total_var) * math.exp(
            -((x - cx) ** 2 + (y - cy) ** 2) / (2.0 * total_var))
    return value

def draw_params(rng):
    return [(rng.uniform(0.25, 0.75), rng.uniform(0.25, 0.75),
             rng.uniform(0.004, 0.010), rng.uniform(0.6, 1.0)) for _ in range(2)]

model = Kratos.Model()
sharp_part = create_tet_cube(model, "Sharp", 6, (Kratos.TEMPERATURE,))
blurred_part = create_tet_cube(model, "Blurred", 6, (Kratos.TEMPERATURE, Kratos.NODAL_ERROR))
print("two parts on the same unit cube:", sharp_part.NumberOfNodes(), "nodes each")


two parts on the same unit cube: 343 nodes each


## Exporting matched (condition, target) grid series

Two `GridDatasetExportProcess` instances — one per part — write step-matched series that
`CreateGridPairDataset` zips into training pairs.


In [2]:
def make_export(part_name, path):
    settings = Kratos.Parameters("""{
        "Parameters": {
            "model_part_name" : "%s",
            "list_of_fields"  : [ { "variable_name" : "TEMPERATURE", "data_location" : "node_historical" } ],
            "grid_shape"      : [16, 16, 2],
            "output_path"     : "%s"
        }
    }""" % (part_name, path))
    process = grid_dataset_export_process.Factory(settings, model)
    process.ExecuteInitialize()
    return process

export_sharp = make_export("Sharp", "output/diffusion_sharp")
export_blurred = make_export("Blurred", "output/diffusion_blurred")

rng = numpy.random.default_rng(0)
N_SAMPLES = 48
for step in range(1, N_SAMPLES + 1):
    params = draw_params(rng)
    for node in sharp_part.Nodes:
        node.SetSolutionStepValue(Kratos.TEMPERATURE, mixture(node.X, node.Y, params))
    for node in blurred_part.Nodes:
        node.SetSolutionStepValue(Kratos.TEMPERATURE, mixture(node.X, node.Y, params, blur=BLUR))
    for part, process in ((sharp_part, export_sharp), (blurred_part, export_blurred)):
        part.ProcessInfo[Kratos.STEP] = step
        part.ProcessInfo[Kratos.TIME] = float(step)
        process.ExecuteFinalizeSolutionStep()

print(N_SAMPLES, "matched (blurred, sharp) pairs exported")


48 matched (blurred, sharp) pairs exported


## Training the conditional denoiser

`EDMPrecondSuperResolution` wraps a `SongUNet` with the EDM preconditioning;
`TrainDiffusionModel` runs physicsnemo's conditional EDM loss over the pairs. Both the
preconditioner and the checkpoint are ordinary physicsnemo `Module`s.


In [3]:
from physicsnemo.diffusion.preconditioners import EDMPrecondSuperResolution

pairs = CreateGridPairDataset("output/diffusion_blurred", "output/diffusion_sharp", squeeze_axis=2)
print(len(pairs), "training pairs")

denoiser = EDMPrecondSuperResolution(
    img_resolution=16, img_in_channels=1, img_out_channels=1,
    model_type="SongUNet", model_channels=16, channel_mult=[1, 2],
    num_blocks=1, attn_resolutions=[])

history = diffusion_utils.TrainDiffusionModel(denoiser, pairs, Kratos.Parameters("""{
    "epochs"        : 120,
    "batch_size"    : 8,
    "learning_rate" : 2e-4,
    "echo_interval" : 40,
    "seed"          : 0
}"""))
print(f"loss: {history[0]:.3e} -> {history[-1]:.3e}")

training_utils.SaveTrainedModel(denoiser, OUTPUT / "downscaler.mdlus", card={
    "input_fields":  [{"variable_name": "TEMPERATURE", "data_location": "node_historical"}],
    "output_fields": [{"variable_name": "TEMPERATURE", "data_location": "node_non_historical"}],
})


48 training pairs
loss: 7.511e-01 -> 3.990e-02


## Deployment: ensemble mean + uncertainty band

On a **held-out** field, `DiffusionInferenceProcess` samples the blurred condition onto the grid,
generates a reverse-diffusion ensemble, and writes the ensemble mean to the output field and the
ensemble standard deviation to `NODAL_ERROR` — a per-node uncertainty for free.


In [4]:
held_out = draw_params(numpy.random.default_rng(1234))
for node in blurred_part.Nodes:
    node.SetSolutionStepValue(Kratos.TEMPERATURE, mixture(node.X, node.Y, held_out, blur=BLUR))

process = diffusion_inference_process.Factory(Kratos.Parameters("""{
    "Parameters": {
        "model_part_name"    : "Blurred",
        "model_settings"     : {
            "checkpoint_file" : "output/downscaler.mdlus",
            "checkpoint_type" : "physicsnemo"
        },
        "input_fields"       : [ { "variable_name" : "TEMPERATURE", "data_location" : "node_historical" } ],
        "output_fields"      : [ { "variable_name" : "TEMPERATURE", "data_location" : "node_non_historical" } ],
        "uncertainty_fields" : [ { "variable_name" : "NODAL_ERROR", "data_location" : "node_non_historical" } ],
        "grid_shape"         : [16, 16, 2],
        "squeeze_axis"       : 2,
        "sampler_settings"   : { "num_samples" : 8, "num_steps" : 12, "seed" : 0 }
    }
}"""), model)

blurred_part.ProcessInfo[Kratos.STEP] = 1
process.ExecuteFinalizeSolutionStep()

truth = numpy.array([mixture(node.X, node.Y, held_out) for node in blurred_part.Nodes])
condition = numpy.array([node.GetSolutionStepValue(Kratos.TEMPERATURE) for node in blurred_part.Nodes])
predicted = numpy.array([node.GetValue(Kratos.TEMPERATURE) for node in blurred_part.Nodes])
spread = numpy.array([node.GetValue(Kratos.NODAL_ERROR) for node in blurred_part.Nodes])

rmse = lambda a: float(numpy.sqrt(numpy.mean((a - truth) ** 2)))
print(f"rmse(blurred condition vs truth) = {rmse(condition):.4f}")
print(f"rmse(ensemble mean vs truth)     = {rmse(predicted):.4f}")
print(f"mean uncertainty (ensemble std)  = {float(spread.mean()):.4f}")


rmse(blurred condition vs truth) = 0.0542
rmse(ensemble mean vs truth)     = 0.0300
mean uncertainty (ensemble std)  = 0.0357


## Variations

The process never asks what the condition channels *mean* — only the training pairs change:

- **Downscaling (CorrDiff)**: condition = coarse solve, target = fine field (this notebook).
- **Generative design (TopoDiff-style)**: condition = constraint masks, target = the design field.
- **Flow reconstruction**: condition = sparse masked observations, target = the full field.

More sampler quality: raise `num_steps` (18 is the EDM default) and `num_samples`; the ensemble
spread narrows as the model sharpens.
